# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**Nombre:** David Pilataxi  
**Fecha:** 13/11/2026

In [1]:
# Dependencias (ejecutar una sola vez)
%pip install -q nltk scikit-learn pandas numpy scipy jinja2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

**Resultados esperados – Ejercicio 1**

* Una tabla `docs` con una fila por libro y las columnas `doc_id | title | text | tokens` (IDs como cadenas), con el mismo preprocesamiento de la práctica 03 (minúsculas, solo letras, sin stopwords, tokens de más de 2 letras).
* $N=18$ documentos y un vocabulario de **41 071 términos**; longitudes $|d|$ con mínimo 3 536, media $avgdl\approx 49\,592$ y máximo 329 877 tokens (los mismos valores de la práctica 03).
* $df(t)\in[1,18]$: los términos presentes en todos los libros tienen $idf_{TF\text{-}IDF}=1$ y el IDF BM25 más bajo; los términos de un solo libro (p. ej. `stubb`, `queequeg`) tienen $df=1$ y $idf_{TF\text{-}IDF}=\ln 18+1\approx 3.89$.
* Estas estadísticas se calculan una sola vez y se reutilizan en todas las consultas.

In [2]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize

try: gutenberg.fileids()                       # descarga el corpus solo si falta
except LookupError: nltk.download("gutenberg")

# Preprocesamiento de la práctica anterior: minúsculas, solo letras, sin stopwords y sin tokens de <3 letras
def prep(texto):
    return [w for w in re.findall(r"[a-z]+", texto.lower())
            if w not in ENGLISH_STOP_WORDS and len(w) > 2]

titulos = {"austen-emma.txt":"Emma", "austen-persuasion.txt":"Persuasion", "austen-sense.txt":"Sense and Sensibility",
 "bible-kjv.txt":"The King James Bible", "blake-poems.txt":"Poems (Blake)", "bryant-stories.txt":"Stories to Tell to Children",
 "burgess-busterbrown.txt":"The Adventures of Buster Bear", "carroll-alice.txt":"Alice's Adventures in Wonderland",
 "chesterton-ball.txt":"The Ball and the Cross", "chesterton-brown.txt":"The Wisdom of Father Brown",
 "chesterton-thursday.txt":"The Man Who Was Thursday", "edgeworth-parents.txt":"The Parent's Assistant",
 "melville-moby_dick.txt":"Moby Dick", "milton-paradise.txt":"Paradise Lost", "shakespeare-caesar.txt":"Julius Caesar",
 "shakespeare-hamlet.txt":"Hamlet", "shakespeare-macbeth.txt":"Macbeth", "whitman-leaves.txt":"Leaves of Grass"}

# Tabla doc_id | title | text | tokens (IDs como cadenas)
docs = pd.DataFrame({"doc_id": [str(i) for i in gutenberg.fileids()],
                     "title": [titulos[i] for i in gutenberg.fileids()],
                     "text": [gutenberg.raw(i) for i in gutenberg.fileids()]})
docs["tokens"] = docs["text"].apply(prep)

# Matriz dispersa documento x término con las frecuencias f(t,d)
cv = CountVectorizer(analyzer=lambda toks: toks)  # la entrada ya está tokenizada
C = cv.fit_transform(docs["tokens"]).tocsr()
vocab = cv.vocabulary_                            # término -> columna

# Estadísticas de la colección (se calculan una sola vez)
N = C.shape[0]
df = np.asarray((C > 0).sum(axis=0)).ravel()      # nº de documentos que contienen el término
idf_tfidf = np.log(N / df) + 1                    # IDF de la práctica 03 (ln(N/DF)+1)
idf_bm25 = np.log(1 + (N - df + 0.5) / (df + 0.5))  # IDF de BM25 (siempre > 0)
dl = np.asarray(C.sum(axis=1)).ravel()            # |d|
avgdl = dl.mean()

terms = pd.DataFrame({"term": cv.get_feature_names_out(), "df": df,
                      "idf_tfidf": idf_tfidf.round(3), "idf_bm25": idf_bm25.round(3)})
print(f"N={N} documentos, {len(vocab)} términos, |d| mín/media/máx = {dl.min():.0f} / {avgdl:.1f} / {dl.max():.0f}")
display(docs[["doc_id", "title"]].assign(longitud=dl))
display(terms.sort_values(["df", "term"], ascending=[False, True]).head(5))  # más frecuentes
display(terms.sort_values(["df", "term"]).head(5))                           # más raros

N=18 documentos, 41071 términos, |d| mín/media/máx = 3536 / 49592.3 / 329877


,doc_id,title,longitud
0,austen-emma.txt,Emma,60045
1,austen-persuasion.txt,Persuasion,31913
2,austen-sense.txt,Sense and Sensibility,44955
3,bible-kjv.txt,The King James Bible,329877
4,blake-poems.txt,Poems (Blake),3536
5,bryant-stories.txt,Stories to Tell to Children,19222
6,burgess-busterbrown.txt,The Adventures of Buster Bear,6710
7,carroll-alice.txt,Alice's Adventures in Wonderland,10622
8,chesterton-ball.txt,The Ball and the Cross,34741
9,chesterton-brown.txt,The Wisdom of Father Brown,30974


,term,df,idf_tfidf,idf_bm25
773,age,18,1.0,0.027
1396,angry,18,1.0,0.027
2575,away,18,1.0,0.027
3235,bed,18,1.0,0.027
3601,best,18,1.0,0.027


,term,df,idf_tfidf,idf_bm25
1,aaronites,1,3.89,2.539
3,abaddon,1,3.89,2.539
4,abaft,1,3.89,2.539
5,abagtha,1,3.89,2.539
6,abana,1,3.89,2.539


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

**Resultados esperados – Ejercicio 2**

* Una tabla con las 8 consultas en inglés (idioma del corpus, ya sin stopwords), la necesidad de información de cada una y el $df$ de cada término, que justifica la mezcla de términos muy frecuentes, raros y consultas de 1 a 6 términos.
* Para cada consulta, dos listas ordenadas por score descendente y `doc_id` ascendente (como cadena), con **a lo sumo 10** documentos con score positivo por método.
* Las consultas con pocos documentos coincidentes devuelven listas de menos de 10 elementos.
* La consulta sin términos en el vocabulario (`zzyzx`) produce **dos listas vacías**.
* Las estadísticas ($N$, $df$, $idf$, $|d|$, $avgdl$) no cambian entre consultas; solo cambia la consulta.

In [3]:
K1, B = 1.2, 0.75

# Consultas y qué busca el usuario con cada una
queries = {
    "whale": "Quiere libros que hablen de ballenas (un solo término raro).",
    "man time good": "Busca textos generales sobre el hombre y el tiempo (términos muy frecuentes, presentes en casi todos los libros).",
    "white whale captain ahab": "Busca la novela del capitán Ahab y la ballena blanca (mezcla de raros y comunes).",
    "rabbit queen garden tea": "Busca el cuento infantil de la reina, el conejo y el té (términos poco frecuentes).",
    "love death": "Busca temas de amor y muerte en poesía y teatro (dos términos frecuentes).",
    "king david lord": "Busca pasajes bíblicos sobre el rey David (términos de frecuencia media).",
    "satan angels heaven fall hell sin": "Busca la caída de los ángeles y el infierno (consulta larga, 6 términos).",
    "zzyzx": "Término inexistente en el vocabulario (listas vacías).",
}

def qtoks(q):
    return prep(q)  # mismo preprocesamiento que los documentos

# Justificación con df
display(pd.DataFrame(
    [(q, need, ", ".join(f"{t}:{df[vocab[t]]}" for t in qtoks(q) if t in vocab) or "(ninguno)")
     for q, need in queries.items()],
    columns=["consulta", "necesidad", f"df por término (N={N})"]).style.set_properties(**{"text-align": "left"}))

# --- TF-IDF + coseno (matriz dispersa, filas normalizadas L2, calculada una vez) ---
X = normalize(C.multiply(idf_tfidf).tocsr())

def cosine(q):
    qv = normalize(cv.transform([qtoks(q)]).multiply(idf_tfidf).tocsr())  # términos fuera del vocabulario se ignoran
    return (X @ qv.T).toarray().ravel()

# --- BM25: aporte de cada término de la consulta a cada documento ---
def bm25_terms(q, k1=K1, b=B):
    ts = [t for t in dict.fromkeys(qtoks(q)) if t in vocab]      # términos únicos presentes en el vocabulario
    idx = [vocab[t] for t in ts]
    f = C[:, idx].toarray()                                      # f(t,d)
    norm = k1 * (1 - b + b * dl / avgdl)                         # normalización por longitud
    return ts, idf_bm25[idx] * f * (k1 + 1) / (f + norm[:, None])

def bm25(q, k1=K1, b=B):
    return bm25_terms(q, k1, b)[1].sum(axis=1)

# --- Ranking: score desc, doc_id asc (cadenas), solo scores positivos ---
def top(scores, n=10):
    cand = [(d, s) for d, s in zip(docs["doc_id"], scores) if s > 1e-12]
    return sorted(cand, key=lambda x: (-round(x[1], 12), x[0]))[:n]

results = {q: (top(cosine(q)), top(bm25(q))) for q in queries}
for q, (t, bm) in results.items():
    print(f"{q!r}\n  TF-IDF ({len(t)}): {[d for d, _ in t]}\n  BM25   ({len(bm)}): {[d for d, _ in bm]}")

,consulta,necesidad,df por término (N=18)
0,whale,Quiere libros que hablen de ballenas (un solo término raro).,whale:6
1,man time good,"Busca textos generales sobre el hombre y el tiempo (términos muy frecuentes, presentes en casi todos los libros).","man:18, time:18, good:18"
2,white whale captain ahab,Busca la novela del capitán Ahab y la ballena blanca (mezcla de raros y comunes).,"white:16, whale:6, captain:9, ahab:2"
3,rabbit queen garden tea,"Busca el cuento infantil de la reina, el conejo y el té (términos poco frecuentes).","rabbit:6, queen:15, garden:15, tea:9"
4,love death,Busca temas de amor y muerte en poesía y teatro (dos términos frecuentes).,"love:15, death:18"
5,king david lord,Busca pasajes bíblicos sobre el rey David (términos de frecuencia media).,"king:15, david:3, lord:15"
6,satan angels heaven fall hell sin,"Busca la caída de los ángeles y el infierno (consulta larga, 6 términos).","satan:7, angels:11, heaven:13, fall:18, hell:11, sin:9"
7,zzyzx,Término inexistente en el vocabulario (listas vacías).,(ninguno)


'whale'
  TF-IDF (6): ['melville-moby_dick.txt', 'bryant-stories.txt', 'whitman-leaves.txt', 'shakespeare-hamlet.txt', 'chesterton-ball.txt', 'bible-kjv.txt']
  BM25   (6): ['melville-moby_dick.txt', 'bryant-stories.txt', 'whitman-leaves.txt', 'shakespeare-hamlet.txt', 'chesterton-ball.txt', 'bible-kjv.txt']
'man time good'
  TF-IDF (10): ['edgeworth-parents.txt', 'chesterton-brown.txt', 'whitman-leaves.txt', 'bryant-stories.txt', 'melville-moby_dick.txt', 'austen-persuasion.txt', 'milton-paradise.txt', 'chesterton-thursday.txt', 'chesterton-ball.txt', 'austen-emma.txt']
  BM25   (10): ['edgeworth-parents.txt', 'austen-emma.txt', 'austen-persuasion.txt', 'melville-moby_dick.txt', 'austen-sense.txt', 'bryant-stories.txt', 'bible-kjv.txt', 'whitman-leaves.txt', 'chesterton-thursday.txt', 'chesterton-ball.txt']
'white whale captain ahab'
  TF-IDF (10): ['melville-moby_dick.txt', 'austen-persuasion.txt', 'bryant-stories.txt', 'whitman-leaves.txt', 'chesterton-brown.txt', 'blake-poems.txt',

**Interpretación – Ejercicio 2**

Los resultados muestran que el comportamiento de los dos métodos depende de la frecuencia de los términos y de la longitud de los documentos. Para la consulta `whale`, ambos métodos devuelven los mismos seis documentos y mantienen el mismo orden; *Moby Dick* ocupa claramente el primer lugar. Esto es coherente con la presencia del término en la obra de Melville y con su capacidad para distinguir documentos dentro de la colección.

En `man time good`, los tres términos aparecen en los 18 documentos, por lo que tienen poca capacidad discriminativa: su IDF de BM25 es bajo. TF-IDF y BM25 coinciden en el primer documento, pero difieren en el resto del ranking. BM25 produce puntuaciones muy próximas entre sí, mientras que el coseno TF-IDF distribuye más los resultados según la frecuencia relativa de los términos en cada documento.

Las consultas más específicas también muestran coincidencias en el primer resultado: *Moby Dick* para `white whale captain ahab`, *Alice's Adventures in Wonderland* para `rabbit queen garden tea`, *Leaves of Grass* para `love death`, la Biblia para `king david lord` y *Paradise Lost* para `satan angels heaven fall hell`. Sin embargo, el orden de los siguientes documentos cambia entre métodos. Esto evidencia que compartir términos no implica asignarles la misma importancia: BM25 incorpora saturación de frecuencia y normalización por longitud, mientras que TF-IDF + coseno compara vectores ponderados y normalizados.

Finalmente, `zzyzx` no contiene ningún término del vocabulario. Por ello, ambos métodos generan listas vacías, tal como se esperaba. Los valores numéricos de coseno y BM25 pertenecen a escalas distintas, por lo que es adecuado comparar el orden de los documentos dentro de cada método, no sus puntuaciones absolutas entre métodos.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

**Resultados esperados – Ejercicio 3**

* Por cada consulta, una tabla `Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25` (con celdas vacías si una lista es más corta).
* Debajo de cada tabla: $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$, el tamaño real de cada lista y el primer documento de cada método. En la consulta vacía (`zzyzx`) el solapamiento es 0 y ambas listas están vacías.
* Puede haber IDs comunes ordenados de forma distinta entre ambos métodos.
* Una tabla final con un par de documentos cuya distancia en el ranking cambia más entre métodos, con $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de la consulta. Se espera que la diferencia se explique por la **longitud** del documento (BM25 penaliza los libros largos), la **saturación** de $f(t,d)$ o la **rareza** del término.

In [4]:
import itertools
title = docs.set_index("doc_id")["title"]

def cell(lst, i, with_title=True):
    if i >= len(lst):
        return ""
    return f"{lst[i][0]} – {title[lst[i][0]]}" if with_title else round(lst[i][1], 4)

# Tabla comparativa por consulta
for q, (t, bm) in results.items():
    n = max(len(t), len(bm))
    tab = pd.DataFrame({"Posición": range(1, n + 1),
                        "ID y título TF-IDF": [cell(t, i) for i in range(n)],
                        "Coseno": [cell(t, i, False) for i in range(n)],
                        "ID y título BM25": [cell(bm, i) for i in range(n)],
                        "Score BM25": [cell(bm, i, False) for i in range(n)]})
    comun = {d for d, _ in t} & {d for d, _ in bm}
    print(f"\nConsulta {q!r}: |T_TFIDF ∩ T_BM25| = {len(comun)} | tamaños: TF-IDF={len(t)}, BM25={len(bm)} | "
          f"primero: TF-IDF={t[0][0] if t else None}, BM25={bm[0][0] if bm else None}")
    if n:
        display(tab)

# Par de documentos cuya distancia en el ranking cambia más entre métodos
best = None
for q, (t, bm) in results.items():
    pt = {d: i for i, (d, _) in enumerate(t)}
    pb = {d: i for i, (d, _) in enumerate(bm)}
    for a, b_ in itertools.combinations([d for d in pt if d in pb], 2):
        ch = abs((pt[a] - pt[b_]) - (pb[a] - pb[b_]))
        if best is None or ch > best[0]:
            best = (ch, q, a, b_)

if best:
    _, qsel, da, db = best
    t, bm = results[qsel]
    pt = {d: i + 1 for i, (d, _) in enumerate(t)}
    pb = {d: i + 1 for i, (d, _) in enumerate(bm)}
    ts, contrib = bm25_terms(qsel)
    st, sb = dict(t), dict(bm)                      # score de cada método por documento
    filas = []
    for d in (da, db):
        i = docs.index[docs["doc_id"] == d][0]
        for j, term in enumerate(ts):
            filas.append({"doc_id": d, "pos TF-IDF": pt[d], "coseno": round(st[d], 4), "pos BM25": pb[d],
                          "score BM25": round(sb[d], 4), "|d|": int(dl[i]),
                          "|d|/avgdl": round(dl[i] / avgdl, 2), "término": term,
                          "f(t,d)": int(C[i, vocab[term]]), "df(t)": int(df[vocab[term]]),
                          "aporte BM25": round(contrib[i, j], 4)})
    print(f"\nConsulta elegida: {qsel!r} | avgdl={avgdl:.0f}")
    display(pd.DataFrame(filas))
else:
    print("Ninguna consulta tiene dos documentos comunes a ambos top 10.")


Consulta 'whale': |T_TFIDF ∩ T_BM25| = 6 | tamaños: TF-IDF=6, BM25=6 | primero: TF-IDF=melville-moby_dick.txt, BM25=melville-moby_dick.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt – Moby Dick,0.5210,melville-moby_dick.txt – Moby Dick,2.3559
1,2,bryant-stories.txt – Stories to Tell to Children,0.0268,bryant-stories.txt – Stories to Tell to Children,2.2678
2,3,whitman-leaves.txt – Leaves of Grass,0.0051,whitman-leaves.txt – Leaves of Grass,1.8538
3,4,shakespeare-hamlet.txt – Hamlet,0.0030,shakespeare-hamlet.txt – Hamlet,1.8453
4,5,chesterton-ball.txt – The Ball and the Cross,0.0014,chesterton-ball.txt – The Ball and the Cross,1.6105
5,6,bible-kjv.txt – The King James Bible,0.0002,bible-kjv.txt – The King James Bible,0.7623



Consulta 'man time good': |T_TFIDF ∩ T_BM25| = 8 | tamaños: TF-IDF=10, BM25=10 | primero: TF-IDF=edgeworth-parents.txt, BM25=edgeworth-parents.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,edgeworth-parents.txt – The Parent's Assistant,0.1655,edgeworth-parents.txt – The Parent's Assistant,0.1752
1,2,chesterton-brown.txt – The Wisdom of Father Brown,0.1626,austen-emma.txt – Emma,0.1752
2,3,whitman-leaves.txt – Leaves of Grass,0.1571,austen-persuasion.txt – Persuasion,0.1750
3,4,bryant-stories.txt – Stories to Tell to Children,0.1328,melville-moby_dick.txt – Moby Dick,0.1749
4,5,melville-moby_dick.txt – Moby Dick,0.1259,austen-sense.txt – Sense and Sensibility,0.1748
5,6,austen-persuasion.txt – Persuasion,0.1078,bryant-stories.txt – Stories to Tell to Children,0.1748
6,7,milton-paradise.txt – Paradise Lost,0.1038,bible-kjv.txt – The King James Bible,0.1748
7,8,chesterton-thursday.txt – The Man Who Was Thur...,0.0998,whitman-leaves.txt – Leaves of Grass,0.1747
8,9,chesterton-ball.txt – The Ball and the Cross,0.0940,chesterton-thursday.txt – The Man Who Was Thur...,0.1746
9,10,austen-emma.txt – Emma,0.0900,chesterton-ball.txt – The Ball and the Cross,0.1745



Consulta 'white whale captain ahab': |T_TFIDF ∩ T_BM25| = 9 | tamaños: TF-IDF=10, BM25=10 | primero: TF-IDF=melville-moby_dick.txt, BM25=melville-moby_dick.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt – Moby Dick,0.5574,melville-moby_dick.txt – Moby Dick,8.6237
1,2,austen-persuasion.txt – Persuasion,0.0798,bible-kjv.txt – The King James Bible,6.6899
2,3,bryant-stories.txt – Stories to Tell to Children,0.0244,bryant-stories.txt – Stories to Tell to Children,3.7252
3,4,whitman-leaves.txt – Leaves of Grass,0.0212,whitman-leaves.txt – Leaves of Grass,3.5822
4,5,chesterton-brown.txt – The Wisdom of Father Brown,0.0194,shakespeare-hamlet.txt – Hamlet,2.1177
5,6,blake-poems.txt – Poems (Blake),0.0145,chesterton-ball.txt – The Ball and the Cross,1.9162
6,7,edgeworth-parents.txt – The Parent's Assistant,0.0128,austen-persuasion.txt – Persuasion,1.7845
7,8,bible-kjv.txt – The King James Bible,0.0107,edgeworth-parents.txt – The Parent's Assistant,1.7758
8,9,chesterton-thursday.txt – The Man Who Was Thur...,0.0071,chesterton-brown.txt – The Wisdom of Father Brown,1.7643
9,10,chesterton-ball.txt – The Ball and the Cross,0.0064,chesterton-thursday.txt – The Man Who Was Thur...,1.4164


... (6 outputs left out)


Consulta 'satan angels heaven fall hell sin': |T_TFIDF ∩ T_BM25| = 10 | tamaños: TF-IDF=10, BM25=10 | primero: TF-IDF=milton-paradise.txt, BM25=milton-paradise.txt


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,milton-paradise.txt – Paradise Lost,0.2334,milton-paradise.txt – Paradise Lost,6.4914
1,2,blake-poems.txt – Poems (Blake),0.0321,bible-kjv.txt – The King James Bible,6.1682
2,3,bible-kjv.txt – The King James Bible,0.0262,chesterton-ball.txt – The Ball and the Cross,5.8096
3,4,whitman-leaves.txt – Leaves of Grass,0.0199,chesterton-thursday.txt – The Man Who Was Thur...,5.3924
4,5,chesterton-ball.txt – The Ball and the Cross,0.0145,melville-moby_dick.txt – Moby Dick,5.0946
5,6,melville-moby_dick.txt – Moby Dick,0.0110,whitman-leaves.txt – Leaves of Grass,4.8411
6,7,chesterton-thursday.txt – The Man Who Was Thur...,0.0101,chesterton-brown.txt – The Wisdom of Father Brown,3.3466
7,8,shakespeare-macbeth.txt – Macbeth,0.0095,blake-poems.txt – Poems (Blake),2.6440
8,9,chesterton-brown.txt – The Wisdom of Father Brown,0.0080,shakespeare-hamlet.txt – Hamlet,2.0326
9,10,shakespeare-hamlet.txt – Hamlet,0.0070,shakespeare-macbeth.txt – Macbeth,2.0118



Consulta 'zzyzx': |T_TFIDF ∩ T_BM25| = 0 | tamaños: TF-IDF=0, BM25=0 | primero: TF-IDF=None, BM25=None

Consulta elegida: 'man time good' | avgdl=49592


,doc_id,pos TF-IDF,coseno,pos BM25,score BM25,|d|,|d|/avgdl,término,"f(t,d)",df(t),aporte BM25
0,whitman-leaves.txt,3,0.1571,8,0.1747,58668,1.18,man,243,18,0.0583
1,whitman-leaves.txt,3,0.1571,8,0.1747,58668,1.18,time,172,18,0.0582
2,whitman-leaves.txt,3,0.1571,8,0.1747,58668,1.18,good,141,18,0.0581
3,austen-emma.txt,10,0.0900,2,0.1752,60045,1.21,man,236,18,0.0583
4,austen-emma.txt,10,0.0900,2,0.1752,60045,1.21,time,280,18,0.0584
5,austen-emma.txt,10,0.0900,2,0.1752,60045,1.21,good,358,18,0.0584


**Interpretación – Ejercicio 3**

Los rankings presentan coincidencias importantes, aunque no son idénticos. En las consultas con resultados, ambos métodos colocan el mismo documento en primer lugar: *Moby Dick* para `whale` y `white whale captain ahab`, *Alice's Adventures in Wonderland* para `rabbit queen garden tea`, *Leaves of Grass* para `love death`, la Biblia para `king david lord` y *Paradise Lost* para `satan angels heaven fall hell`. El solapamiento de los top 10 varía entre 6 y 10 documentos; alcanza 10 para la consulta sobre Satanás, ángeles e infierno. En `zzyzx`, ambos conjuntos están vacíos.

La mayor diferencia relativa de orden seleccionada por el análisis corresponde a `man time good`, al comparar `whitman-leaves.txt` con `austen-emma.txt`. En TF-IDF, *Leaves of Grass* ocupa la posición 3 (coseno 0.1571), mientras que *Emma* queda en la posición 10 (0.0900). En BM25 el orden se invierte: *Emma* alcanza la posición 2 (0.1752) y *Leaves of Grass* queda en la posición 8 (0.1747).

Los tres términos (`man`, `time` y `good`) aparecen en los 18 documentos, por lo que `df(t)=18` para cada uno y su IDF BM25 es bajo. Las frecuencias en *Leaves of Grass* son 243, 172 y 141; en *Emma* son 236, 280 y 358, respectivamente. Los aportes BM25 son aproximadamente 0.058 por término en ambos libros. La frecuencia mayor de `time` y `good` en *Emma* compensa la pequeña diferencia de longitud: sus 60 045 tokens equivalen a 1.21 veces `avgdl`, frente a 58 668 tokens (1.18 veces `avgdl`) en *Leaves of Grass*. Debido a la saturación de la frecuencia, las diferencias de aporte son pequeñas y los scores BM25 quedan muy próximos.

En cambio, TF-IDF + coseno favorece a *Leaves of Grass* para esta consulta, según la distribución relativa de los tres términos en su vector normalizado. El ejemplo demuestra que BM25 no solo cuenta apariciones: combina frecuencia, IDF y longitud del documento. También confirma que una consulta formada por términos muy comunes tiene menor poder para separar documentos que una consulta con términos más específicos.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

**Resultados esperados – Ejercicio 4**

* Una tabla con los 5 primeros documentos del ranking base, más el documento coincidente más corto y el más largo, con su posición (y score) en las tres configuraciones: base ($k_1=1.2$, $b=0.75$), sin ajuste por longitud ($b=0$) y mayor efecto de repeticiones ($k_1=2.0$).
* Con $b=0$ se espera que los documentos **largos** suban (ya no se penalizan) y los cortos bajen.
* Con $k_1=2.0$ se espera que el aporte crezca más para los documentos con $f(t,d)$ **alta**, porque la saturación llega más tarde.
* Si ningún documento cambia de posición, se imprimen los aportes por término de cada configuración para ver que el score cambió pero no el orden relativo.

In [5]:
Q4 = qsel if best else "white whale captain ahab"  # consulta con coincidencias en documentos de distinta longitud (puedes cambiarla)
configs = {"Base": (1.2, 0.75), "b=0": (1.2, 0.0), "k1=2.0": (2.0, 0.75)}

pos, sc = {}, {}
for name, (k1, b) in configs.items():
    sc[name] = bm25(Q4, k1, b)
    pos[name] = {d: i + 1 for i, (d, _) in enumerate(top(sc[name], None))}  # ranking completo

match = list(pos["Base"])                                    # documentos con coincidencias
por_len = sorted(match, key=lambda d: dl[docs.index[docs["doc_id"] == d][0]])
sel = list(dict.fromkeys(match[:5] + [por_len[0], por_len[-1]]))  # top 5 base + más corto y más largo
ii = [docs.index[docs["doc_id"] == d][0] for d in sel]
tab = pd.DataFrame({"doc_id": sel, "|d|": dl[ii].astype(int)})
for name in configs:
    tab[name] = [f"{pos[name].get(d)} ({sc[name][i]:.3f})" for d, i in zip(sel, ii)]
print(f"Consulta: {Q4!r}  (posición y score BM25)")
display(tab)

# Si el orden no cambia, comparar los aportes por término
if all(pos[n][d] == pos["Base"][d] for n in configs for d in sel):
    for name, (k1, b) in configs.items():
        ts, c = bm25_terms(Q4, k1, b)
        print(f"\nAportes por término – {name}")
        display(pd.DataFrame(c[ii], index=sel, columns=ts).round(4))

Consulta: 'man time good'  (posición y score BM25)


,doc_id,|d|,Base,b=0,k1=2.0
0,edgeworth-parents.txt,67068,1 (0.175),2 (0.175),1 (0.238)
1,austen-emma.txt,60045,2 (0.175),4 (0.175),2 (0.238)
2,austen-persuasion.txt,31913,3 (0.175),7 (0.175),3 (0.238)
3,melville-moby_dick.txt,95943,4 (0.175),3 (0.175),4 (0.237)
4,austen-sense.txt,44955,5 (0.175),6 (0.175),5 (0.237)
5,blake-poems.txt,3536,18 (0.153),18 (0.123),18 (0.194)
6,bible-kjv.txt,329877,7 (0.175),1 (0.176),7 (0.237)


**Interpretación – Ejercicio 4**

La consulta utilizada fue `man time good`. Al eliminar la normalización por longitud (`b=0`), los cambios más notorios aparecen en documentos de longitud extrema. La Biblia, con 329 877 tokens, pasa de la posición 7 en la configuración base a la posición 1; al dejar de penalizar los documentos largos, su puntuación aumenta relativamente frente a los demás. En el extremo opuesto, *Poems (Blake)*, con solo 3 536 tokens, permanece en la posición 18 y su score baja de 0.153 a 0.123. Esto concuerda con el efecto esperado de retirar el ajuste que favorece documentos más cortos que el promedio y penaliza los más largos.

Los documentos de longitud intermedia también cambian de posición al usar `b=0`: *Emma* pasa del puesto 2 al 4 y *Persuasion* del 3 al 7, mientras que *Moby Dick* sube del 4 al 3. Por tanto, el parámetro `b` sí modifica el orden de recuperación, no solamente la escala de las puntuaciones.

Al aumentar `k1` de 1.2 a 2.0, las posiciones de los siete documentos mostrados se mantienen iguales que en la configuración base. Sin embargo, los scores aumentan: por ejemplo, *The Parent's Assistant* pasa de 0.175 a 0.238 y *Poems (Blake)* de 0.153 a 0.194. Un `k1` mayor hace que la saturación de la frecuencia sea más gradual, de modo que las repeticiones adicionales conservan más influencia antes de alcanzar el límite. En esta consulta, ese cambio aumenta las puntuaciones, pero no es suficiente para alterar el orden relativo de los documentos seleccionados.

En conclusión, `b` controla el efecto de la longitud respecto a `avgdl`, mientras que `k1` controla la rapidez con que se satura la contribución de las frecuencias. Los resultados observados son consistentes con ambos efectos: `b=0` beneficia especialmente al documento más largo y `k1=2.0` eleva los scores sin cambiar las posiciones de los documentos examinados.